# Reciprocal-Space Mapping

The public RSM path separates a source, persisted geometry, and an
`RSMPlan`. Smoke mode creates a bounded `RSMVolume` for slice review;
real mode requires the matching `PixelQMap`, authenticated physical UB,
and scan motor mapping. HKL is the established/default frame; Cartesian
Q must be selected explicitly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import RSMPlan, run_rsm
from xrd_tools.io import open_scan
from xrd_tools.rsm import RSMCoordinateFrame, RSMVolume
from xrd_tools.viz import plot_image


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
processed_file = TEST_DATA / "processed.nexus"
mapper = None  # Real mode: PixelQMap from the experiment's persisted geometry.
UB = None  # Real mode: authenticated physical 3x3 UB for this source.
diff_motors = ()  # Real mode: one persisted scan_data motor name per circle.
slice_axis = widgets.Dropdown(options=("h", "k", "l"), value="l", description="integrate")
compute = widgets.Button(description="Compute RSM", button_style="primary")
status = widgets.HTML("<i>Compute is explicit; changing a slice does not regrid data.</i>")
output = widgets.Output()
display(widgets.VBox([slice_axis, compute, status, output]))


In [ ]:
NOTEBOOK_STATE = {"computes": 0, "slice_draws": 0, "volume": None}

def _synthetic_volume():
    h, k, l = np.linspace(-0.08, 0.08, 24), np.linspace(-0.06, 0.06, 20), np.linspace(0.90, 1.10, 18)
    hh, kk, ll = np.meshgrid(h, k, l, indexing="ij")
    return RSMVolume(h, k, l, np.exp(-0.5 * ((hh / 0.02) ** 2 + (kk / 0.018) ** 2 + ((ll - 1.0) / 0.03) ** 2)))

def draw_cached_slice(_=None):
    volume = NOTEBOOK_STATE["volume"]
    if volume is None:
        return
    with output:
        clear_output(wait=True)
        axis_a, axis_b, image, integrated = volume.get_slice(slice_axis.value)
        fig, ax = plt.subplots(figsize=(6, 4))
        plot_image(ax, image.T, attrs={"xlabel": "axis 1", "ylabel": "axis 2", "title": f"RSM projection over {slice_axis.value}"}, cb_label="Intensity")
        plt.show()
        display({"shape": volume.shape, "integrated_points": len(integrated), "bounds": volume.get_bounds()})
        NOTEBOOK_STATE["slice_draws"] += 1

def compute_rsm(_=None):
    try:
        if SMOKE_MODE:
            volume = _synthetic_volume()
        else:
            assert processed_file.is_file(), f"Missing processed NeXus: {processed_file}"
            assert mapper is not None and diff_motors, "Set mapper and diff_motors from the experiment geometry."
            assert UB is not None, "Set the authenticated physical UB for this source."
            ub = np.asarray(UB, dtype=np.float64)
            assert ub.shape == (3, 3) and np.all(np.isfinite(ub)), "UB must be a finite 3x3 matrix."
            volume = run_rsm(
                RSMPlan(
                    mapper=mapper,
                    diff_motors=tuple(diff_motors),
                    bins=(96, 96, 96),
                    UB=ub,
                    coordinate_frame=RSMCoordinateFrame.HKL,
                ),
                open_scan(processed_file),
            ).payload
        NOTEBOOK_STATE.update(computes=NOTEBOOK_STATE["computes"] + 1, volume=volume)
        status.value = "<b>RSM gridding complete; slice changes redraw cached volume only.</b>"
        draw_cached_slice()
    except Exception as exc:
        status.value = f"<b>RSM compute failed:</b> {exc}"
        raise

compute.on_click(compute_rsm)
slice_axis.observe(draw_cached_slice, names="value")
NOTEBOOK_ACTIONS = {"compute_rsm": compute_rsm, "draw_cached_slice": draw_cached_slice}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    compute_rsm()
